# PCA Lab

Three applications of PCA. Work in your group, and talk through *how* to do each
one before you write any code — the method is the exercise.

Two words we will keep using. **Projecting** a point means casting its shadow:
onto PC1 alone the shadow lands somewhere on a line, onto PC1 and PC2 it lands
somewhere on a plane. **Rebuilding** draws that shadow back in the original
space — the average point, plus each component multiplied by that point's score
on it. Application 1 uses the shadow. Applications 2 and 3 draw it back and
compare.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

---

## Application 1 · Can PCA be used as a classifier?

This data holds measurements taken from tissue samples, and each sample has been
diagnosed as malignant or benign. Project the samples onto PC1 alone, then onto
PC1 and PC2, and look at where the two diagnoses fall. PCA is never shown the
diagnosis, so any separation you see came out of the measurements by themselves.

*Make sure you scale the columns first.*

**Can PCA be used as a classifier? If so, how — and how well does it do?**

In [ ]:
from sklearn.datasets import load_breast_cancer



---

## Application 2 · Shrinking an image

A photograph is just a grid of numbers, so you can treat one as a dataset: each
row of the image is a data point and each column across is a feature. Run PCA on
those rows, keep the first k components, and rebuild the picture from them.

*Make sure you convert the photo to grayscale first.*

**Which is the smallest k that still gives a picture you would call good enough?
And how much storage did you save — counting everything you would need in order
to rebuild it?**

In [ ]:
from sklearn.datasets import load_sample_image



---

## Application 3 · Finding what does not fit

PCA rebuilds each data point from a few components, and most points rebuild well
because they follow the pattern those components describe. A point that rebuilds
badly does not follow it. Use that to find the districts in this housing data
that do not fit, then look at them and work out what makes them unusual.

*Make sure you scale the columns first.*

**Which districts are the strangest, and what makes them strange?**

In [ ]:
from sklearn.datasets import fetch_california_housing



---

# Explore more

Not part of the lab. Something to try on your own if the idea interests you.

### The components as the answer

In the three applications above, the components were a means to an end. Sometimes
they are the result you wanted.

Banks charge a different interest rate depending on how long you borrow, and this
data records how ten of those rates changed each day. Run PCA and look at the
first few components: each one is a pattern describing how the ten rates tend to
move together. Plot each component's values across the loan lengths and describe
in plain words what it does.

The cell below tries real published rates and falls back to simulated data if the
download does not work. Either way the exercise is the same.

In [ ]:
maturities = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]   # years you borrow for
series = ["DGS3MO","DGS6MO","DGS1","DGS2","DGS3","DGS5","DGS7","DGS10","DGS20","DGS30"]

try:
    import pandas as pd
    cols = {}
    for s in series:
        url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + s
        cols[s] = pd.read_csv(url, index_col=0, parse_dates=True).iloc[:, 0]
    rates = pd.DataFrame(cols).apply(pd.to_numeric, errors="coerce").dropna()
    changes = rates.diff().dropna().to_numpy()
    print("real data:", changes.shape, "days x maturities")
except Exception as e:
    print("download unavailable (" + type(e).__name__ + "), using simulated data instead")
    rng = np.random.default_rng(0)
    m = np.array(maturities)
    lg = np.log(m) - np.log(m).mean()
    shapes = [np.ones_like(m), lg / np.abs(lg).max(), None]
    bend = -(lg ** 2); bend -= bend.mean(); shapes[2] = bend / np.abs(bend).max()
    sizes = [1.0, 0.5, 0.25]
    changes = sum(np.outer(rng.normal(0, s, 1000), sh) for s, sh in zip(sizes, shapes))
    changes = changes + rng.normal(0, 0.05, changes.shape)
    print("simulated data:", changes.shape, "days x maturities")

# your code here


### Other places PCA turns up

- **Recommender systems.** Decompose a table of users against items and the
  components behave like tastes nobody defined. Real systems do not use plain PCA,
  because most of the table is empty and PCA needs a complete one, but the idea of
  a few hidden factors is the same.
- **Genetics recovering geography.** PCA on 1,387 Europeans genotyped at 197,146
  sites, plotted on the first two components, comes out looking like a map of
  Europe. No geographic information was given to it.
- **Faces.** The same trick as Application 2, applied to photographs of faces.
  The components look like ghostly average faces, and early face recognition
  compared people's coordinates in that space.

A warning that covers all of them. PCA only builds straight combinations of your
columns and chooses them by spread alone, so when the structure you care about is
curved, or lives in a direction that barely varies, PCA will miss it. Methods
that handle those cases exist — kernel PCA, t-SNE, UMAP — and they all begin from
the question this unit began from: what would a good new feature even be?